In [3]:
import awswrangler as wr
import pandas as pd

#database_name = table_name = "perturbation_evaluations_17_year"
    # Read the table metadata and data using the Glue catalo

#df_pert = wr.s3.read_parquet_table(
#    table=table_name,
#    database=database_name
#
df_pert = pd.read_parquet('s3://jdinvestment/perturbation_evaluations_17_year/164cc309ceff4afd8b5078ad975ca332.snappy.parquet')
PERTURBATION_FOLDER = "s3://jdinvestment/perturbations_17_year"
df_parents = pd.read_csv("{}/perturbations.csv".format(PERTURBATION_FOLDER))
df_pert = df_pert.set_index('sim_id').join(df_parents[['sim_id', 'parent_id']].set_index('sim_id'))

In [12]:
df_pert.columns

Index(['worst_annual_return', 'drawdown_integral', 'annualized_return',
       'worst_annual_4wk', 'dollar_ret_1p', 'dollar_ret_6p', 'dollar_ret_13p',
       'dollar_ret_26p', 'avg_eps_1q', 'avg_eps_2q', 'avg_eps_4q',
       'avg_eps_8q', 'threshold', 'beta', 'mom_decay', 'qual_decay',
       'macro_weights_0', 'macro_weights_1', 'macro_weights_2',
       'macro_weights_3', 'parent_id'],
      dtype='str')

In [4]:
df_mean = df_pert[['parent_id', 'drawdown_integral', 'annualized_return', 'worst_annual_4wk']].groupby('parent_id').agg('mean')
df_std = df_pert[['parent_id', 'drawdown_integral', 'annualized_return', 'worst_annual_4wk']].groupby('parent_id').agg('std')
df_mean.columns = ['drawdown_mean', 'return_mean', 'worst_annual_mean']
df_std.columns = ['drawdown_std', 'return_std', 'worst_annual_std']
df = df_mean.join(df_std).sort_values(by = 'return_mean', ascending=False)
df

,drawdown_mean,return_mean,worst_annual_mean,drawdown_std,return_std,worst_annual_std
parent_id,,,,,,
4.464538e+09,49985.867348,0.194315,-0.028235,16114.774505,0.010238,0.001643
5.641167e+09,51999.943622,0.193973,-0.028218,19343.608480,0.009666,0.001831
4.570857e+09,57532.204798,0.193771,-0.028573,21541.946859,0.009472,0.002195
8.153094e+09,55190.414768,0.191844,-0.027887,19479.551993,0.009180,0.001681
5.674948e+09,49438.372641,0.191199,-0.028478,16722.294466,0.010616,0.002407
...,...,...,...,...,...,...
6.029063e+09,64452.755960,0.135979,-0.034590,10619.027674,0.008633,0.002727
3.860909e+09,59927.712787,0.134002,-0.035208,8345.885284,0.007009,0.002986
8.751513e+09,47435.364804,0.133293,-0.035847,8326.484732,0.008008,0.002402


In [10]:
import plotly.graph_objects as go  

fig = go.Figure()
fig.add_trace(go.Scatter(x = df.drawdown_mean, y = df.return_mean, mode = 'markers'))
fig.show()

In [18]:
import pandas as pd
import numpy as np

def get_pareto_frontier(df, objective_cols, senses):
    """
    Returns a subset of the dataframe containing only the Pareto optimal rows.
    
    Parameters:
    - df: The input DataFrame (e.g., your Pymoo results)
    - objective_cols: List of column names to evaluate (e.g., ['drawdown', 'return'])
    - senses: List of strings ('min' or 'max') corresponding to each column
    """
    # 1. Normalize all objectives to 'minimize' to simplify logic
    # We multiply 'max' objectives by -1
    data = df[objective_cols].to_numpy(copy=True)
    for i, sense in enumerate(senses):
        if sense == 'max':
            data[:, i] *= -1

    n_rows = data.shape[0]
    is_pareto = np.ones(n_rows, dtype=bool)

    # 2. Vectorized Domination Check
    for i in range(n_rows):
        if is_pareto[i]:
            # A row is dominated if another row is <= in all objectives 
            # AND < in at least one objective.
            # Here we check if row 'i' is dominated by any other row
            is_dominated = np.all(data <= data[i], axis=1) & np.any(data < data[i], axis=1)
            
            # If anyone dominates 'i', 'i' is not Pareto
            if np.any(is_dominated):
                is_pareto[i] = False
            else:
                # Optimization: If 'i' is Pareto, it might dominate others. 
                # Mark those it dominates as not Pareto.
                others_dominated = np.all(data[i] <= data, axis=1) & np.any(data[i] < data, axis=1)
                is_pareto[others_dominated] = False

    return df.iloc[is_pareto].copy()

# --- Example Usage ---
objectives = ['drawdown_mean', 'return_mean']
senses = ['min', 'max']
pareto_df = get_pareto_frontier(df, objectives, senses)
print(f"Found {len(pareto_df)} Pareto optimal solutions.")

Found 4 Pareto optimal solutions.


In [19]:
pareto_df.columns

Index(['drawdown_mean', 'return_mean', 'worst_annual_mean', 'drawdown_std',
       'return_std', 'worst_annual_std'],
      dtype='str')

In [36]:
conservative_parent_id = int(pareto_df.index[pareto_df.drawdown_mean.argmin()])
conservative_parent_id
df_cons = pd.DataFrame(df_pert.loc[df_pert.parent_id == conservative_parent_id].mean(0)).transpose()
df_cons.to_csv('../analysis/perturbations_17_year_conservative_star.csv', index = False)
df_cons

/tmp/ipykernel_4317/4266598782.py:3: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  df_cons = pd.DataFrame(df_pert.loc[df_pert.parent_id == conservative_parent_id].mean(0)).transpose()


,worst_annual_return,drawdown_integral,annualized_return,worst_annual_4wk,dollar_ret_1p,dollar_ret_6p,dollar_ret_13p,dollar_ret_26p,avg_eps_1q,avg_eps_2q,...,avg_eps_8q,threshold,beta,mom_decay,qual_decay,macro_weights_0,macro_weights_1,macro_weights_2,macro_weights_3,parent_id
0,-0.113206,31496.178455,0.14,-0.025081,-1.875184,0.083033,-1.086411,1.250082,1.628416,0.868194,...,-1.050469,-0.1145,14.514457,0.959481,0.92588,-0.827745,0.364693,-0.086431,0.72195,4.017874e+09


In [19]:
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x = pareto_df.drawdown_mean, y = pareto_df.return_mean, mode = 'markers'))
fig2.show()

In [20]:
pareto_df = pareto_df.rename(columns =  dict(zip(['drawdown_mean', 'return_mean', 'worst_annual_mean'], [ 'drawdown_integral', 'annualized_return', 'worst_annual_4wk'])))
pareto_df.to_csv('../analysis/pareto_17_year_stars.csv', index = 'False')
pareto_df

,drawdown_integral,annualized_return,worst_annual_4wk,drawdown_std,return_std,worst_annual_std
parent_id,,,,,,
4.464538e+09,49985.867348,0.194315,-0.028235,16114.774505,0.010238,0.001643
5.674948e+09,49438.372641,0.191199,-0.028478,16722.294466,0.010616,0.002407
8.493154e+09,37694.871294,0.142834,-0.030636,10291.297566,0.009281,0.002217
4.017874e+09,31496.178455,0.140000,-0.025081,6592.330978,0.006741,0.002108


In [31]:
df_weights = df_parents.set_index('parent_id').loc[pareto_df.index].reset_index().groupby('parent_id').agg('mean')
df_weights.drop([col for col in df_weights if 'unnamed' in col.lower()], axis = 1, inplace = True)
df_out = df_weights.join(pareto_df)
df_out.to_csv('../analysis/pareto_17_year_stars.csv', index = False)